In [3]:
import sys, os
REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, os.path.join(REPO, 'Code', 'tools'))
from pathlib import Path
import pandas as pd
import local_utility_functions as luf

# Change root directory to the repo root (Jupyter: __file__ is not defined)
def find_repo_root(start=Path.cwd()):
	for p in [start] + list(start.parents):
		if (p / 'Code').exists() or (p / '.git').exists() or (p / 'local_repo').exists():
			return p
	return start

repo_root = find_repo_root()
data_root = repo_root.parent.parent / 'Data' / 'LIS' / 'code'
mail_root = Path('/Users/jedrek/Library/Mail/V10')

os.chdir(repo_root)
sys.path.append(str(repo_root / 'Code' / 'tools'))


In [4]:
df_voiv = pd.read_csv(data_root.parent / "LIS_Voiv_2000.csv")
df_groups = pd.read_csv(data_root.parent / "LIS_Groups_2000.csv")

---
## Append additional rows from .txt files

The helper `append_txt_dir(df, directory, block_type)` scans every `.txt` file in `directory`, parses all `data_region_...` or `data_group_...` blocks (same Python-dict format as the email output), and merges the new rows into the existing DataFrame.

`block_type` is either `"voiv"` (default) or `"group"`.  
Columns present only in one side are filled with `NaN`.  
Duplicate `(year, region)` pairs are resolved by keeping whatever is already in the DataFrame (new rows only fill genuinely missing year/region combinations).


In [5]:
import ast
import re

# ── Low-level parsers (identical to LIS01_reader) ───────────────────────────

def _parse_value(raw: str):
    """Convert a bracketed scalar literal like [1986] or ["N/A"] to a Python scalar."""
    raw = raw.strip()
    try:
        val = ast.literal_eval(raw)
        if isinstance(val, list) and len(val) == 1:
            v = val[0]
            return None if (isinstance(v, str) and v in ("N/A", "NA")) else v
        return val
    except Exception:
        return None


def _parse_dict_body(body: str) -> dict:
    """Parse key-value lines inside a data_... = { … } block."""
    out = {}
    for m in re.finditer(r"'([^']+)'\s*:\s*(\[[^\]]*\])", body):
        out[m.group(1)] = _parse_value(m.group(2))
    return out


# ── Block-level regex (covers both voiv and group naming conventions) ────────

_VOIV_RE = re.compile(
    r"data_region_(\d{4})_(\w+)\s*=\s*\{(.+?)\n\}",
    re.DOTALL,
)
_GROUP_RE = re.compile(
    r"data_group_(\d{4})_(Group_\d+)\s*=\s*\{(.+?)\n\}",
    re.DOTALL,
)


def parse_txt_blocks(path, block_type: str = "voiv") -> list[dict]:
    """
    Parse a single .txt file and return a list of flat row dicts.

    Parameters
    ----------
    path : str or Path
        Path to the .txt file.
    block_type : {"voiv", "group"}
        "voiv"  → data_region_YEAR_REGION blocks; region kept as-is.
        "group" → data_group_YEAR_Group_NNN blocks; region formatted as "Group NNN".
    """
    text = Path(path).read_text(encoding="utf-8", errors="replace")
    regex = _VOIV_RE if block_type == "voiv" else _GROUP_RE
    rows = []
    for m in regex.finditer(text):
        year_str, raw_name, body = m.group(1), m.group(2), m.group(3)
        row = _parse_dict_body(body)
        row["year"]   = int(year_str)
        row["region"] = raw_name if block_type == "voiv" else raw_name.replace("_", " ", 1)
        rows.append(row)
    return rows


def append_txt_dir(df: pd.DataFrame, directory, block_type: str = "voiv") -> pd.DataFrame:
    """
    Scan every .txt file in *directory*, parse all matching blocks, and append
    the new (year, region) rows to *df*.

    - Columns present only in the txt data are added (filled with NaN in old rows).
    - Columns present only in df are filled with NaN for the new rows.
    - Duplicate (year, region) pairs already in df are silently skipped.

    Returns the combined, sorted DataFrame (does not modify df in-place).
    """
    new_rows = []
    for txt_path in sorted(Path(directory).glob("*.txt")):
        new_rows.extend(parse_txt_blocks(txt_path, block_type=block_type))

    if not new_rows:
        print("No blocks found in txt files.")
        return df

    df_new = pd.DataFrame(new_rows)

    # Coerce numeric columns
    skip_cols = {"year", "region"}
    for col in df_new.columns:
        if col not in skip_cols:
            df_new[col] = pd.to_numeric(df_new[col], errors="coerce")

    # Combine, drop duplicates keeping the rows already in df (keep="first"
    # after putting df first), then re-sort
    combined = (
        pd.concat([df, df_new], ignore_index=True)
        .drop_duplicates(subset=["year", "region"], keep="first")
        .sort_values(["year", "region"])
        .reset_index(drop=True)
    )
    print(f"Loaded {len(new_rows)} rows from {sum(1 for _ in Path(directory).glob('*.txt'))} file(s). "
          f"Added {len(combined) - len(df)} new row(s) to DataFrame.")
    return combined


In [6]:
ADDITIONAL_OUTPUTS = repo_root.parent.parent / "Data" / "LIS" / "additional_outputs"

df_voiv = append_txt_dir(df_voiv, ADDITIONAL_OUTPUTS, block_type="voiv")

print(f"\ndf_voiv shape  : {df_voiv.shape}")
print(f"Years covered  : {sorted(df_voiv['year'].unique())}")
print(f"Regions (sample): {sorted(df_voiv['region'].unique())[:10]}")


Loaded 67 rows from 3 file(s). Added 67 new row(s) to DataFrame.

df_voiv shape  : (403, 200)
Years covered  : [np.int64(1986), np.int64(1992), np.int64(1995), np.int64(1999), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Regions (sample): ['1', '2', '3', '4', '5', '6', '7', '8', '9', 'bialostockie']


In [7]:
df_voiv['region_id'] = df_voiv['region']

lis_macroregions_dict = {
    1: "Central",
    2: "North-East",
    3: "North",
    4: "South",
    5: "South-East",
    6: "Middle-East",
    7: "Middle",
    8: "Middle-West",
    9: "South-West"
}

for idx, row in df_voiv.iterrows():
    try:
        region_id = int(row['region'])
    except ValueError:
        region_id = None
    if region_id is not None and region_id in lis_macroregions_dict.keys():
        df_voiv.at[idx, 'region'] = lis_macroregions_dict[region_id]
    
    current_region = df_voiv.at[idx, 'region']
    # Capitalize the first letter of each word in the region name
    df_voiv.at[idx, 'region'] = current_region.title().replace('_', '-')


In [8]:
# Load region dictionaries
new_region_path = data_root.parent.parent / "new_voiv_region_dict.json"
old_region_path = data_root.parent.parent / "old_voiv_region_dict.json"

# Load the region dictionaries
new_region_dict = pd.read_json(new_region_path, typ='series').to_dict()
old_region_dict = pd.read_json(old_region_path, typ='series').to_dict()

for idx, row in df_voiv.iterrows():
    region = row['region']
    if region.lower() in new_region_dict.values() and row['year'] >= 1999:
        df_voiv.at[idx, 'region_id'] = str(list(new_region_dict.keys())[list(new_region_dict.values()).index(region.lower())]).zfill(7)
    elif region.lower() in old_region_dict.values():
        df_voiv.at[idx, 'region_id'] = str(list(old_region_dict.keys())[list(old_region_dict.values()).index(region.lower())])

In [9]:
# Load country data
country_data_path = data_root.parent / 'country level'

lis_h = luf.adapt_txt(file_path= Path(country_data_path / "LIS_h_country.txt"), save_path= Path(country_data_path.parent / "LIS_h_country.csv"))
lis_p = luf.adapt_txt(file_path= Path(country_data_path / "LIS_p_country.txt"), save_path= Path(country_data_path.parent / "LIS_p_country.csv"))

In [10]:
# Join deflation columns to both datasets.
# We create deflator factor for each year. We collect them in 4 columns each for one base year (1986, 1990, 2017, 2023).
# Since we have monthly inflation data, we assmue that the inflation rate for the whole year is the same as the inflation rate in January of that year.

# Load monthly inflation data
# One column: inflation_index containing month to month inflation index (1.02 means 2% inflation compared to the previous month) indexed by date (first day of each month)

inflation_data = pd.read_csv(data_root.parent.parent / 'Auxiliary data' / 'inflation_data_cleaned.csv', index_col=0, parse_dates=True)
inflation_data = inflation_data.sort_index()

# Data starts in 1982-01-01, so we can remove all rows before 1986-01-01 (the first base year)
inflation_data = inflation_data[inflation_data.index >= '1986-01-01']

print(inflation_data.head())
# First value is 102.0. -> We have to divide the index by 100 to get the actual inflation index (1.02 means 2% inflation compared to the previous month)
inflation_data['inflation_index'] = inflation_data['inflation_index'] / 100

col_mapping = {
    1986: 'deflator_1986',
    1990: 'deflator_1990',
    2017: 'deflator_2017',
    2023: 'deflator_2023'
}

            inflation_index
1986-01-01            102.0
1986-02-01            101.5
1986-03-01            102.7
1986-04-01            103.0
1986-05-01            101.6


In [11]:

# ── Step 1: Build cumulative monthly price-level index ────────────────────────
#
# inflation_index[t] = P(t) / P(t-1)  (month-to-month ratio, 1.02 = 2% vs prev month)
# cumprod() gives an absolute (but arbitrarily-scaled) price level for every month.
# Only ratios between entries matter; the absolute scale cancels out.

price_level = inflation_data['inflation_index'].cumprod()   # Timestamp → price level

# ── Step 2: Annual average price level (correct base for yearly survey data) ───
#
# For a YEARLY survey that covers income/wealth over a full calendar year,
# the representative price level for that year is the MEAN of all 12 monthly
# price levels — not just January.  Using January alone would be biased,
# especially during hyperinflation years (e.g. 1989-1991 in Poland).

annual_avg_price = price_level.groupby(price_level.index.year).mean()  # int year → price level

# Price level at January of each base year (the fixed reference point)
jan_base_price = {yr: price_level[pd.Timestamp(f'{yr}-01-01')] for yr in col_mapping}

# ── Step 3: Compute and attach deflators to both DataFrames ──────────────────
#
# deflator(base_year, survey_year) = P(Jan base_year) / avg_annual_P(survey_year)
#
# Usage:  value_in_base_year_Jan_money = value_in_survey_year_money × deflator
#
# Examples (base = 2017):
#   • 1990 had very low prices vs 2017 → deflator >> 1 → 100 PLN (1990) worth >> 100 PLN in 2017 money
#   • 2023 had higher prices than Jan 2017 → deflator < 1 → 100 PLN (2023) worth < 100 PLN in 2017 money

for base_year, col in col_mapping.items():
    if base_year not in jan_base_price or pd.Timestamp(f'{base_year}-01-01') not in price_level.index:
        print(f"[!] No Jan price data for {base_year} — skipping {col}."); continue

    base_p = jan_base_price[base_year]
    yearly_defl = base_p / annual_avg_price      # Series[int year → deflator]

    df_voiv[col]   = df_voiv['year'].map(yearly_defl)
    df_groups[col] = df_groups['year'].map(yearly_defl)
    lis_p[col]     = lis_p['year'].map(yearly_defl)
    lis_h[col]     = lis_h['year'].map(yearly_defl)

# ── Step 4: Spot-check table ─────────────────────────────────────────────────
spot_years = [1986, 1990, 1995, 2000, 2005, 2010, 2017, 2020, 2023]
available_spot = [y for y in spot_years if y in annual_avg_price.index]

print("=== 100 PLN in source year → PLN in base-year money (annual-average price level) ===\n")
header = f"{'Year':>6}" + "".join(f"  base={b:>4}" for b in col_mapping)
print(header); print("-" * len(header))
for y in available_spot:
    row = f"{y:>6}"
    for base_year in col_mapping:
        val = 100 * jan_base_price[base_year] / annual_avg_price[y]
        row += f"  {val:>10.2f}"
    print(row)

# ── Step 5: Sanity checks ─────────────────────────────────────────────────────
print("\n\nSanity checks (base = 2017):")
for src, direction, note in [
    (available_spot[0], ">", f"{available_spot[0]} pre-hyperinflation → >> 100 in 2017 PLN"),
    (2017, "≈", "2017 annual avg ≈ but not exactly 100 (avg price > Jan price due to rising prices)"),
    (available_spot[-1], "<", f"{available_spot[-1]} post-2017 → < 100 in 2017 PLN"),
]:
    if src not in annual_avg_price.index: continue
    v = 100 * jan_base_price[2017] / annual_avg_price[src]
    ok = (direction == ">" and v > 100) or (direction == "<" and v < 100) or (direction == "≈" and 85 < v < 115)
    print(f"  {'✓' if ok else '✗'}  100 PLN ({src}) = {v:>12,.2f} PLN in 2017 money  [{note}]")

print(f"\n  NaNs in df_voiv  : {df_voiv[list(col_mapping.values())].isna().any().to_dict()}")
print(f"  NaNs in df_groups: {df_groups[list(col_mapping.values())].isna().any().to_dict()}")

print("\n\ndf_voiv — deflator columns by year:")
print(
    df_voiv[['year'] + list(col_mapping.values())]
    .drop_duplicates('year')
    .sort_values('year')
    .to_string(index=False)
)


=== 100 PLN in source year → PLN in base-year money (annual-average price level) ===

  Year  base=1986  base=1990  base=2017  base=2023
--------------------------------------------------
  1986       92.40     3208.19    76469.73   108029.43
  1990        1.90       65.94     1571.85     2220.56
  1995        0.32       11.01      262.42      370.72
  2000        0.17        6.05      144.15      203.64
  2005        0.15        5.28      125.84      177.77
  2010        0.13        4.58      109.23      154.31
  2017        0.12        4.17       99.49      140.54
  2020        0.11        3.88       92.50      130.67
  2023        0.08        2.90       69.04       97.53


Sanity checks (base = 2017):
  ✓  100 PLN (1986) =    76,469.73 PLN in 2017 money  [1986 pre-hyperinflation → >> 100 in 2017 PLN]
  ✓  100 PLN (2017) =        99.49 PLN in 2017 money  [2017 annual avg ≈ but not exactly 100 (avg price > Jan price due to rising prices)]
  ✓  100 PLN (2023) =        69.04 PLN in 2017

In [12]:
# Save both files again to Data/replication_package

df_voiv.to_csv(data_root.parent.parent / "replication_package" / "LIS" / 'LIS_Voiv.csv', index=False)
df_groups.to_csv(data_root.parent.parent / "replication_package" / "LIS" / 'LIS_Groups.csv', index=False)
lis_country = pd.merge(lis_h, lis_p, on=['year'], how='outer', suffixes=('_h', '_p'))
lis_country.to_csv(data_root.parent.parent / "replication_package" / "LIS" / 'LIS_Country.csv', index=False)
